## Task 10: End to End Similarity Search Pipeline

In [15]:
from langchain_community.document_loaders import TextLoader,DirectoryLoader,PyPDFLoader,WebBaseLoader,CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.vectorstores import FAISS

In [18]:
def Search_Pipeline(source,q_value,k_value=3,embedding_model=1,vector=1):
    docs = []
    if source.startswith(("http://", "https://")):
        docs = WebBaseLoader(source).load()
        splitter = RecursiveCharacterTextSplitter.from_language(language=Language.HTML,chunk_size=100,chunk_overlap=0)
    else:
        txt_docs = DirectoryLoader(source,glob=['**/*.txt','**/*.md'],loader_cls=TextLoader,loader_kwargs={'autodetect_encoding': True}).load()
        pdf_docs = DirectoryLoader(source,glob='**/*.pdf',loader_cls=PyPDFLoader).load()
        csv_docs = DirectoryLoader(source,glob='**/*.csv',loader_cls=CSVLoader).load()
        docs = txt_docs + pdf_docs + csv_docs
        splitter = RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=0)
    chunks = splitter.split_documents(docs)
    #Selecting Embedding Model:
    if embedding_model==1:
        embedding = OllamaEmbeddings(model='nomic-embed-text')
    else:
        embedding = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

    #Selecting Vector DataBase Type:
    if vector == 1:
        vector_store = Chroma(embedding_function=embedding,persist_directory='Chroma_db')
        vector_store.add_documents(chunks)
    else:
        vector_store = FAISS.from_documents(chunks,embedding)

    #Similarity Search
    result = vector_store.similarity_search(query=q_value,k=k_value)
    return result

Search_Pipeline(data,query,k,embedding model,vector db type)
Embedding Mode:
1) 1 = OllamaEmbedding
2) 2 = HuggingFaceEmbedding
Vector DB Type:
1) 1 = ChromaDB
2) 2 = FAISS

In [25]:
result = Search_Pipeline('../data','Afganistan',4,2,2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4065.36it/s]


In [26]:
result

[Document(id='05213ff8-589b-443d-aca5-570a509c5596', metadata={'source': '..\\data\\obama.md'}, page_content='country.'),
 Document(id='6c4413ee-50c3-4d60-bede-5ca833747c24', metadata={'source': '..\\data\\obama.md'}, page_content='His administration authorized the 2011 operation in Pakistan that killed Osama bin Laden, the'),
 Document(id='ab8b0964-e394-4f57-a38a-6bfc458b778f', metadata={'source': '..\\data\\obama.md'}, page_content='specializing in international relations.'),
 Document(id='2f7295d7-7667-4589-9e7e-9d766d7726e0', metadata={'source': '..\\data\\obama.md'}, page_content='leader of al-Qaeda responsible for the September 11, 2001, terrorist attacks.')]

In [19]:
web_result = Search_Pipeline('https://www.obamalibrary.gov/obamas/president-barack-obama','Who is Obama',7,2,2)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1923.98it/s]


In [27]:
web_result

[Document(id='4bc9b378-6ee9-4076-8d3a-4a687911d3fd', metadata={'source': 'https://www.obamalibrary.gov/obamas/president-barack-obama', 'title': 'President Barack Obama | Barack Obama Presidential Library', 'language': 'en'}, page_content='4, 2008, Obama became the first African-American to be elected President. He resigned his seat in th'),
 Document(id='3e9299ba-f986-4aad-a44b-e0e51c403a3c', metadata={'source': 'https://www.obamalibrary.gov/obamas/president-barack-obama', 'title': 'President Barack Obama | Barack Obama Presidential Library', 'language': 'en'}, page_content='Obama was elected to the Illinois State Senate from the thirteenth district. As a State Senator, he'),
 Document(id='c178c914-5e5e-4744-b00c-fddea4c2f198', metadata={'source': 'https://www.obamalibrary.gov/obamas/president-barack-obama', 'title': 'President Barack Obama | Barack Obama Presidential Library', 'language': 'en'}, page_content='e U.S. Senate on November 16, 2008.\nBarack Obama was inaugurated as the 44t